# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring a Croissant dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their fields, using their `@id` for all references.

In [ ]:
# Enumerate all record sets and their fields using their @id
print('Record sets (@id and name):')
rs_ids = [rs['@id'] for rs in metadata.to_json().get('recordSet', [])]
if len(rs_ids) == 0:
    print('No top-level record sets found in metadata. Attempting to extract via distribution.')
    # As per Croissant spec, data is usually linked in distributions, which include record sets.
    distributions = metadata.to_json().get('distribution', [])
    for dist in distributions:
        if 'recordSet' in dist:
            for rs in dist['recordSet']:
                recset = rs if isinstance(rs, dict) else {'@id': rs}
                print(f"- {recset.get('@id', 'unknown')}")
else:
    for rs in metadata.to_json()['recordSet']:
        if isinstance(rs, dict):
            print(f"- {rs['@id']} ({rs.get('name', 'no name')})")

# For this dataset, try to load record set IDs from the distributions, as recordSet is otherwise empty
distribution_obj = metadata.to_json().get('distribution', [])
record_set_ids = []
for d in distribution_obj:
    if isinstance(d, dict) and 'recordSet' in d:
        rs_list = d['recordSet']
        if isinstance(rs_list, list):
            for r in rs_list:
                if isinstance(r, dict) and '@id' in r:
                    record_set_ids.append(r['@id'])
                elif isinstance(r, str):
                    record_set_ids.append(r)

if len(record_set_ids) == 0:
    print('No record set IDs found via distribution either.')
# Display fields for each discovered record set
for recset_id in record_set_ids:
    print(f"\nFields for record set {recset_id}:")
    recset = None
    # try to get record set object
    for d in distribution_obj:
        if 'recordSet' in d:
            for rs in d['recordSet']:
                if (isinstance(rs, dict) and rs.get('@id') == recset_id) or (isinstance(rs, str) and rs == recset_id):
                    recset = rs if isinstance(rs, dict) else None
    if recset and 'field' in recset:
        fields = recset['field']
        for field in fields:
            if isinstance(field, dict):
                fid = field.get('@id', 'unknown')
                fname = field.get('name', 'no name')
                print(f"- {fid}: {fname}")
            elif isinstance(field, str):
                print(f"- {field}")
    else:
        print('No fields available (may need Croissant schema exploration for deeper field listing).')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All references to record sets and fields are via their `@id`.

In [ ]:
# Use record_set_ids discovered in the previous step
if len(record_set_ids) == 0:
    # Fallback: guess from typical Croissant structure
    record_set_ids = ['cr:MainRecordSet']

dataframes = {}
print(f"Extracting and loading record sets: {record_set_ids}")
# We'll load only the first record set for demonstration
for record_set_id in record_set_ids:
    try:
        records_iter = dataset.records(record_set=record_set_id)
        df_records = list(records_iter)
        if len(df_records) > 0:
            df = pd.DataFrame(df_records)
            dataframes[record_set_id] = df
            print(f"Loaded record set {record_set_id}: shape = {df.shape}")
        else:
            print(f"Record set {record_set_id} is empty.")
    except Exception as e:
        print(f"Error extracting record set {record_set_id}: {e}")

# For demonstration, print columns of the first loaded DataFrame
if len(dataframes) > 0:
    first_recset = list(dataframes.keys())[0]
    print(f"Columns in record set {first_recset}:")
    print(dataframes[first_recset].columns.tolist())
    display(dataframes[first_recset].head())
else:
    print('No dataframes loaded. Check the record set IDs and dataset availability.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping by attributes. 

All fields/columns are referenced by their `@id`, as found in the previous step.

In [ ]:
# We'll use the first loaded DataFrame and select a numeric field by its ID (or column name)
if len(dataframes) > 0:
    df_eda = list(dataframes.values())[0]
    recset_id = list(dataframes.keys())[0]
    print(f"Working with record set: {recset_id}")
    # Try to find a numeric field; let's assume field IDs like 'cr:MW' (molecular weight) or similar
    example_numeric_ids = [
        'cr:MW',   # molecular weight
        'cr:pI',   # isoelectric point
        'cr:Coverage',
        'cr:SpectralCount',
        'cr:NormalizedAbundance',
        'cr:PeptideCount'        # some typical column names/ids
    ]
    found_numeric = None
    for col in df_eda.columns:
        if col in example_numeric_ids:
            found_numeric = col
            break
        # Try fuzzy matching
        for eid in example_numeric_ids:
            if col.lower() == eid.lower():
                found_numeric = col
                break
        if found_numeric: break
    if not found_numeric:
        # fallback: try to use first numeric-looking column
        num_cols = df_eda.select_dtypes('number').columns
        if len(num_cols) > 0:
            found_numeric = num_cols[0]
            print(f"No standard field id found; using column {found_numeric}")
        else:
            print("No numeric columns available for EDA.")
    if found_numeric:
        # Filtering: values > threshold
        threshold = df_eda[found_numeric].quantile(0.5) # median as an arbitrary threshold
        filtered_df = df_eda[df_eda[found_numeric] > threshold]
        print(f"Filtered records with {found_numeric} > {threshold:.2f} (median):")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{found_numeric}_normalized"] = (filtered_df[found_numeric] - filtered_df[found_numeric].mean()) / filtered_df[found_numeric].std()
        print(f"Normalized {found_numeric} for filtered records:")
        display(filtered_df[[found_numeric, f"{found_numeric}_normalized"]].head())

        # Try to group by a categorical field, e.g., sample/condition
        example_group_ids = [
            'cr:Sample', 'cr:Condition', 'cr:Group', 'cr:Source', 'cr:Experiment', 'cr:SampleLabel'
        ]
        group_field = None
        for ef in example_group_ids:
            if ef in df_eda.columns:
                group_field = ef
                break
        if not group_field:
            # fallback: first object dtype with less than 10 unique values
            object_cols = df_eda.select_dtypes('object').columns
            for col in object_cols:
                if df_eda[col].nunique() < 10:
                    group_field = col
                    break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[found_numeric].mean()
            print(f"Grouped mean {found_numeric} by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable group_field found for grouping.")
else:
    print("No dataframes available for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field and its normalized version. All field references are by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) > 0:
    df_eda = list(dataframes.values())[0]
    recset_id = list(dataframes.keys())[0]
    # Assume variable selection from EDA above
    # Let's check if found_numeric and filtered_df are in the environment
    try:
        # Distribution plot
        plt.figure(figsize=(8,4))
        sns.histplot(df_eda[found_numeric], kde=True, bins=30)
        plt.title(f"Distribution of {found_numeric} in record set {recset_id}")
        plt.xlabel(found_numeric)
        plt.show()

        if f"{found_numeric}_normalized" in filtered_df.columns:
            plt.figure(figsize=(8,4))
            sns.histplot(filtered_df[f"{found_numeric}_normalized"], kde=True, bins=30)
            plt.title(f"Normalized {found_numeric} (filtered)")
            plt.xlabel(f"{found_numeric}_normalized")
            plt.show()
    except Exception as e:
        print(f"Visualization failed: {e}")
else:
    print("No data available for visualization.")

## 6. Conclusion
In this notebook, we loaded and explored the `Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells` dataset using the `mlcroissant` library via its Croissant schema specification. We demonstrated how to:

- Load a dataset and view its metadata.
- List and reference record sets, fields, and columns by their Croissant `@id`.
- Extract records into DataFrames and perform basic filtering and normalization by numeric fields (referenced by `@id`).
- Group and plot data distributions, all using reproducible data science practices anchored by the Croissant schema.

For deeper analysis, repeat these steps with other record sets and expand EDA and visualization as required by your research goals.